In [ ]:
from egm.datastore.observation_store_memory import InMemoryObservationStore
from egm.services.queries.observation_query_service import ObservationQueryService


def main() -> None:
    store = InMemoryObservationStore()
    svc = ObservationQueryService(store=store)

    # Hand-crafted minimal persistence dicts
    r1 = {
        "task_id": "t1",
        "plan_id": "p1",
        "protocol": "XEB",
        "depth": 3,
        "shots": 10,
        "backend_name": "Ideal",
        "chip_name": "chip-alpha",
        "execution_status": "ok",
        "analysis_status": "ok",
        "execution_error": None,
        "analysis_error": None,
        "observation_time": None,
        "qubits": [0, 1],
        "execution_summary": {"num_pairs": 2, "pairs_digest": None, "artifact_ref": None},
        "analysis_payload": {"xeb_analysis": {}, "spb_analysis": {}, "axis_mode": "depth"},
    }
    r2 = {
        **r1,
        "task_id": "t2",
        "protocol": "RB",
        "depth": 5,
        "chip_name": "chip-beta",
        "execution_summary": {"num_pairs": 0, "pairs_digest": None, "artifact_ref": None},
        "analysis_payload": {},
    }
    r3 = {
        **r1,
        "task_id": "t3",
        "plan_id": "p2",
        "protocol": "XEB",
        "backend_name": "NoisySim",
        "chip_name": "chip-alpha",
        "execution_status": "execution_error",
        "analysis_status": "analysis_error",
        "execution_error": "boom",
        "analysis_error": "nope",
        "execution_summary": {"num_pairs": 0, "pairs_digest": None, "artifact_ref": None},
        "analysis_payload": {},
    }

    id1 = store.save_observation(r1)
    id2 = store.save_observation(r2)
    id3 = store.save_observation(r3)

    # get_observation: hit
    got1 = svc.get_observation(id1)
    assert got1 == r1

    # get_observation: missing id
    missing = svc.get_observation("does-not-exist")
    assert missing is None

    # list_observations: contains all (order not guaranteed)
    all_obs = svc.list_observations()
    assert len(all_obs) == 3
    assert {o["task_id"] for o in all_obs} == {"t1", "t2", "t3"}

    # filter: single field
    only_xeb = svc.filter_observations(protocol="XEB")
    assert {o["task_id"] for o in only_xeb} == {"t1", "t3"}

    # filter: multi-field AND
    xeb_alpha = svc.filter_observations(protocol="XEB", chip_name="chip-alpha")
    assert {o["task_id"] for o in xeb_alpha} == {"t1", "t3"}

    # filter: tighter AND
    xeb_alpha_ideal = svc.filter_observations(
        protocol="XEB", chip_name="chip-alpha", backend_name="Ideal"
    )
    assert {o["task_id"] for o in xeb_alpha_ideal} == {"t1"}

    # filter: empty result
    none = svc.filter_observations(protocol="QPE")
    assert none == []

    # sanity: ids exist in store
    ids = set(store.list_observation_ids())
    assert {id1, id2, id3}.issubset(ids)

    print("observation query smoke passed")


if __name__ == "__main__":
    main()